In [1]:
# =========================================================
# STAGE 4 — SIMILARITY DETECTION
# =========================================================

import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import warnings
warnings.filterwarnings("ignore")

In [2]:
# =========================================================
# LOAD PROCESSED DATASET
# =========================================================

df = pd.read_csv("processed_reviews.csv")

print("Dataset Loaded")

print()

print("Dataset Shape:")
print(df.shape)

Dataset Loaded

Dataset Shape:
(199996, 20)


In [4]:
# =========================================================
# CHECK IMPORTANT COLUMNS
# =========================================================

required_columns = [
    'reviewText',
    'processed_review',
    'reviewerID',
    'asin'
]

display(df[required_columns].head())

,reviewText,processed_review,reviewerID,asin
0,Bought it for a ballet tutu but it is being wo...,bought ballet tutu worn around house legging r...,A2PAVURT4NOHE1,0000031852
1,I origonally didn't get the item I ordered. W...,origonally didnt get item ordered contacting c...,A1SNLWGLFXD70K,0000031852
2,My daughter and her friends love the colors an...,daughter friend love color way fit last long w...,A3URQ0LXLV46E9,0000031852
3,"Arrived very timely, cute grandbaby loves it. ...",arrived timely cute grandbaby love thing scent...,A1KJ4CVG87QW09,0000031852
4,My little girl just loves to wear this tutu be...,little girl love wear tutu stripe even convinc...,AA9ITO6ZLZW6,0000031852


In [8]:
# =========================================================
# CREATE SMALLER SUBSET
# =========================================================

sample_size = 2000

sample_df = df.sample(
    n=sample_size,
    random_state=42
).reset_index(drop=True)

print("Subset Shape:")
print(sample_df.shape)

Subset Shape:
(2000, 20)


In [12]:
# =========================================================
# HANDLE MISSING PROCESSED REVIEWS
# =========================================================

sample_df['processed_review'] = (
    sample_df['processed_review']
    .fillna('')
)

print("Missing Values Fixed")

print()

print(
    sample_df['processed_review']
    .isna()
    .sum()
)

Missing Values Fixed

0


In [13]:
# =========================================================
# TF-IDF VECTORIZATION
# =========================================================

tfidf = TfidfVectorizer(
    max_features=3000,
    ngram_range=(1,2),
    min_df=2
)

X_similarity = tfidf.fit_transform(
    sample_df['processed_review']
)

print("TF-IDF Shape:")
print(X_similarity.shape)

TF-IDF Shape:
(2000, 3000)


In [14]:
# =========================================================
# COMPUTE COSINE SIMILARITY
# =========================================================

print("Computing Cosine Similarity...")

similarity_matrix = cosine_similarity(
    X_similarity
)

print("Similarity Matrix Shape:")
print(similarity_matrix.shape)

Computing Cosine Similarity...
Similarity Matrix Shape:
(2000, 2000)


In [24]:
# =========================================================
# FIND HIGH SIMILARITY REVIEW PAIRS
# =========================================================

threshold = 0.65

similar_pairs = []

rows = similarity_matrix.shape[0]

for i in range(rows):

    for j in range(i + 1, rows):

        similarity_score = similarity_matrix[i][j]

        if similarity_score >= threshold:

            similar_pairs.append({

                'review_1_index': i,
                'review_2_index': j,
                'similarity_score': similarity_score,

                'review_1': sample_df.iloc[i]['reviewText'],
                'review_2': sample_df.iloc[j]['reviewText'],

                'reviewer_1': sample_df.iloc[i]['reviewerID'],
                'reviewer_2': sample_df.iloc[j]['reviewerID'],

                'product_1': sample_df.iloc[i]['asin'],
                'product_2': sample_df.iloc[j]['asin']
            })

print("Highly Similar Pairs Found:")
print(len(similar_pairs))

Highly Similar Pairs Found:
1


In [25]:
# =========================================================
# CREATE RESULTS DATAFRAME
# =========================================================

similarity_results = pd.DataFrame(similar_pairs)

print("Similarity Results Shape:")
print(similarity_results.shape)

similarity_results.head()

Similarity Results Shape:
(1, 9)


,review_1_index,review_2_index,similarity_score,review_1,review_2,reviewer_1,reviewer_2,product_1,product_2
0,107,1223,1.0,Works great!,Works great!,A1U0T90YMBHQ81,A1BB4635KSKDMV,B0000AY38R,B0000BYE9Q


In [26]:
# =========================================================
# TOP SUSPICIOUS REVIEW PAIRS
# =========================================================

top_pairs = similarity_results.sort_values(
    by='similarity_score',
    ascending=False
)

top_pairs[
    [
        'similarity_score',
        'review_1',
        'review_2'
    ]
].head(10)

,similarity_score,review_1,review_2
0,1.0,Works great!,Works great!


In [27]:
# =========================================================
# SAME PRODUCT ANALYSIS
# =========================================================

same_product = similarity_results[
    similarity_results['product_1']
    ==
    similarity_results['product_2']
]

print("Highly Similar Reviews On Same Product:")
print(len(same_product))

Highly Similar Reviews On Same Product:
0


In [29]:
# =========================================================
# STAGE 4 SUMMARY
# =========================================================

print("=" * 60)

print("STAGE 4 COMPLETED SUCCESSFULLY")

print("=" * 60)

print()

print("Total Reviews Analyzed:")
print(sample_df.shape[0])

print()

print("Highly Similar Review Pairs:")
print(len(similarity_results))

print()

print("Similarity Threshold:")
print(threshold)

print()

print("Stage 4 Insights:")
print("- Near-duplicate review detection")
print("- Repetitive template identification")
print("- Suspicious similarity analysis")
print("- Coordinated review behavior signals")

STAGE 4 COMPLETED SUCCESSFULLY

Total Reviews Analyzed:
2000

Highly Similar Review Pairs:
1

Similarity Threshold:
0.65

Stage 4 Insights:
- Near-duplicate review detection
- Repetitive template identification
- Suspicious similarity analysis
- Coordinated review behavior signals


In [30]:
# =========================================================
# SAVE STAGE 4 RESULTS
# =========================================================

similarity_results.to_csv(
    "similarity_results.csv",
    index=False
)

print("Similarity Results Saved Successfully")

Similarity Results Saved Successfully
